# PARC2026 07: Patch Submission Requirements

採点済みの提出物が `import torch` で `libnvJitLink.so.12` を見つけられずに失敗した場合の修復です。原因は `requirements.txt` の1行だけで、重み・組み立て・測定済みのValidator結果はすべて有効なままです。

本番採点イメージはCUDA 13ベースで、提出物のvenvは `--system-site-packages` 付きで作られます。`torch==2.2.0` が引く `nvidia-*-cu12` は全て `==` 固定なのでvenvへ入りますが、`nvidia-nvjitlink-cu12` だけはtorchが直接要求せず、`nvidia-cusparse-cu12` がバージョン指定なしで要求します。イメージ側（焼き込まれた `jax[cuda12]` 由来）のコピーで要求が満たされてしまい、venvへ入りません。`libcusparse.so.12` のRUNPATHは同じsite-packagesしか見ないため、ロード時に落ちます。

このNotebookはDrive上のArchiveを読み、`requirements.txt` の1エントリだけを差し替えた**別名のArchive**を作ります。**元のArchiveは変更しません。**

Colab Terminalからは次と同じです。

```bash
cd /content/Physical_ai
bash training/openvla_oft_a100/scripts/colab_patch_submission.sh
```

所要はおよそ20〜40分（14GBの読み書きとDriveへの書き戻し）。GPUは不要です。

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

修正済みの `requirements.txt` を持つCommitを取得します。ランタイムが作り直されていれば `/content/Physical_ai` は存在しないので、cloneから始まります。`BRANCH` はmainへ取り込んだ後は `main` にしてください。

最後の `grep` で `nvidia-nvjitlink-cu12==12.1.105` が1行出れば、差し替え元の準備ができています。

In [ ]:
from pathlib import Path

REPO = Path('/content/Physical_ai')
BRANCH = 'claude/repository-review-oz7z17'

if not REPO.exists():
    !git clone --branch {BRANCH} https://github.com/yu37330/Physical_ai.git {REPO}
else:
    !git -C {REPO} fetch origin {BRANCH}
    !git -C {REPO} checkout -B {BRANCH} origin/{BRANCH}
%cd {REPO}
!git log -1 --oneline
!grep nvidia-nvjitlink submission/openvla_oft_offline/requirements.txt

Drive上のArchiveを確認します。既定は `MyDrive/PARC2026/60_submissions/parc2026_track1_openvla_oft_plus.zip` です。別の場所にある場合は次のセルで `SOURCE_ZIP` を指定してください。

In [ ]:
!ls -la /content/drive/MyDrive/PARC2026/60_submissions/

In [ ]:
%cd /content/Physical_ai
!bash training/openvla_oft_a100/scripts/colab_patch_submission.sh

## 提出前の確認

`..._patched.zip` をローカルPCへダウンロードし、`patched_submission_sha256.txt` のSHA256と一致することを確認してから提出してください。ダウンロード先の空き容量が足りない場合は、ブラウザの保存先を別ドライブへ変更します。

### 依存解決の再現確認（任意）

この不具合は `--system-site-packages` 付きvenvでしか再現しません。通常のvenvで `pip install -r requirements.txt` すると `nvidia-nvjitlink-cu12` は自動で入るため、検証になりません。

次のセルは採点環境と同じ条件を作り、**venv側に** `libnvJitLink.so.12` が置かれることを確認します。パスが1行表示されれば合格です。torchと`nvidia-*-cu12`を実際に取得するため、3GB前後・5分ほどかかります。提出を急ぐ場合は飛ばしてかまいません（`colab_patch_submission.sh` が運営Validatorの `--pip-dry-run` を既に通しています）。

In [ ]:
%%bash
# イメージ側に nvjitlink がある状況を作ってから、提出物と同じ条件で解決させる
pip install -q nvidia-nvjitlink-cu12
rm -rf /content/reqcheck
python -m venv --system-site-packages /content/reqcheck
/content/reqcheck/bin/pip install -q -r /content/Physical_ai/submission/openvla_oft_offline/requirements.txt
ls /content/reqcheck/lib/python*/site-packages/nvidia/nvjitlink/lib/libnvJitLink.so.12